# Week 9 — AI-assisted coding, debugging, and a tiny bit of OOP

**Notebook outcomes**

- Use an AI coding assistant (Claude Code, Cursor, Copilot) effectively
- Read a Python traceback and locate the root cause
- Use `try` / `except` and `breakpoint()` / `pdb` to diagnose bugs
- Recognize a simple `@dataclass` so you can read library code
- Decide when AI assistance helps vs. hurts

This week has three related but distinct chunks. They go together
because the #1 reason you want AI assistance is to debug faster, and a
lot of library code you'll read in other courses uses dataclasses.


## Part 1 — AI-assisted coding

There are three flavors of AI coding tool you'll bump into:

| Tool | What it is |
|------|------------|
| **Copilot** (GitHub) | In-editor autocomplete. Suggests the next few lines. |
| **Cursor** | VS Code fork with chat + multi-file edits + agent mode. |
| **Claude Code** (Anthropic) | Terminal-based agent. Reads, writes, runs commands. |

For the camp, any of them is fine. Rice students can usually get free
access to Copilot (GitHub Education) and many students use Claude Code.
Pick one and learn it deeply rather than flip-flopping.


### When AI assistance helps

AI is dramatically helpful for:

- **Boilerplate.** Converting data between formats, writing repetitive
  test cases, scaffolding a function.
- **Unfamiliar libraries.** "Give me the matplotlib snippet for a
  stacked bar chart."
- **Explaining errors.** Paste the traceback; ask what's happening.
- **Translating.** MATLAB → Python, R → Python.

Where it helps *less*:

- **Novel research code.** If the literature doesn't have it, the model
  probably doesn't either.
- **Numerical correctness.** The code will look fine and be subtly
  wrong. Always verify on known cases.
- **Performance.** Models love one-liners that are slow.


### Prompting patterns that work

1. **Tell it the goal, not the code.** Bad: "write a loop that…". Good:
   "compute the year-over-year growth rate of the `gdp` list in
   `data.py`."
2. **Give it context.** Paste the relevant data shape, the error
   message, or the surrounding function. Agents that can read files do
   this themselves; autocomplete ones can't.
3. **Ask for tests.** "Write five assertions that exercise edge cases."
   Then you can verify yourself.
4. **Push back.** If the first answer looks wrong, say why. The second
   answer is usually better.
5. **Don't trust the math.** Always spot-check.


### A classroom example

Imagine you're stuck on:

```python
def cumulative_returns(prices):
    # I want a list where element i is (prices[i] / prices[0]) - 1
    ...
```

A good prompt: *"Complete this function. Prices is a list of floats.
Return a list of the same length where each element is the cumulative
return relative to `prices[0]`. Include two assertions that test it."*

A not-so-good prompt: *"Make it work"*.


### Using AI responsibly in this course

- **Understand the code before you submit it.** If you can't explain
  each line, don't submit it.
- **Cite when asked.** Some assignments will say "solve this without AI
  assistance." Respect that — it's a calibration exercise for you.
- **Your grade is for what you know.** The AI doesn't take the exam.


## Part 2 — Reading tracebacks

Python tracebacks look scary. They're actually one of the most
informative artifacts in the language. Read them from the **bottom up**.


In [ ]:
def yoy_growth(prev, curr):
    return (curr - prev) / prev


def report(prev, curr):
    print("growth:", yoy_growth(prev, curr))


# Let's introduce a bug
try:
    report(0, 100)  # division by zero!
except Exception as e:
    import traceback

    traceback.print_exc()

A traceback has two critical pieces of information, at the bottom:

1. The **exception class** — `ZeroDivisionError`.
2. The **exception message** — `division by zero`.

Everything above is the **call stack** — who called whom to get here,
with file and line numbers. Start at the bottom, read the message, then
walk up until you find the first frame in *your* code (not a library).

This workflow alone will save you dozens of hours over the camp.


### Common exceptions cheat sheet

| Exception | Usual cause |
|-----------|-------------|
| `NameError` | typo or used a variable before defining it |
| `TypeError` | wrong kind of value (e.g., `"7" + 3`) |
| `ValueError` | right type but wrong value (`int("abc")`) |
| `KeyError` | missing dict key |
| `IndexError` | list index out of range |
| `AttributeError` | object doesn't have that method/attribute |
| `ZeroDivisionError` | self-explanatory |
| `ImportError` / `ModuleNotFoundError` | package not installed, or wrong name |
| `FileNotFoundError` | path doesn't exist |


## Part 3 — `try` / `except`

When you expect a failure can happen, catch it explicitly:


In [ ]:
def safe_parse_int(text):
    try:
        return int(text)
    except ValueError:
        return None


print(safe_parse_int("42"))
print(safe_parse_int("abc"))

**Guidelines:**

- Catch the **most specific** exception you can. `except Exception:`
  (or worse, a bare `except:`) swallows everything, including bugs you
  didn't know existed.
- Don't use `try`/`except` for normal control flow. If you know a key
  might not be in a dict, use `d.get(key)`, not `try: d[key]`.
- `raise` without an argument re-raises the current exception. Useful
  inside an `except` block to "do something and bubble up".


In [ ]:
def parse_or_explain(text):
    try:
        return int(text)
    except ValueError:
        print(f"[warn] couldn't parse {text!r}")
        raise  # let the caller decide


# Wrapped so we can see both the warning and the traceback
try:
    parse_or_explain("abc")
except ValueError as e:
    print("caught:", e)

## Part 4 — `breakpoint()` and `pdb`

When reading the traceback isn't enough, step into the code. Drop
`breakpoint()` anywhere you want execution to pause:

```python
def hard_to_debug(values):
    total = 0
    for v in values:
        breakpoint()        # drops you into pdb
        total += v / 2
    return total
```

Inside `pdb` you can:

| Command | Does |
|---------|------|
| `p expr` | print an expression |
| `pp expr` | pretty-print |
| `n` | next line (step over) |
| `s` | step into |
| `c` | continue (run until next breakpoint or end) |
| `l` | list source around the current line |
| `q` | quit |

In Jupyter, `breakpoint()` opens a prompt at the bottom of the cell
output. Type commands, press Enter.

**Pro tip:** even without `breakpoint()`, sprinkling `print(...)`
statements with descriptive labels ("entered loop", `f"v={v}"`) fixes
80% of bugs in under a minute. Don't over-engineer the debugging.


## Part 5 — A tiny bit of OOP (`@dataclass`)

You haven't had a "classes" week, and we won't go deep here. But you
*will* encounter classes in every library from week 12 onward. This
section teaches you just enough to read them.


### A `@dataclass` in 10 seconds

A `@dataclass` is a lightweight way to bundle a few fields under a name.
It's roughly "a named tuple with type hints":


In [ ]:
from dataclasses import dataclass


@dataclass
class Student:
    name: str
    major: str
    gpa: float


alice = Student(name="Alice", major="econ", gpa=3.7)
print(alice)
print(alice.name, alice.gpa)

`@dataclass` auto-generates the `__init__` (constructor), a nice
`repr`, and equality. You can still add methods:


In [ ]:
@dataclass
class Student:
    name: str
    major: str
    gpa: float

    def honors(self) -> bool:
        return self.gpa >= 3.7


alice = Student("Alice", "econ", 3.7)
print(alice.honors())

### Reading library code

When you see something like this in a tutorial:

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)
preds = model.predict(X_new)
```

You now know:

- `LinearRegression` is a class.
- `model` is an instance.
- `fit`, `predict` are methods on that instance.
- State lives inside the object (coefficients, etc.).

That's enough to get started. Classes as a deep topic show up in the
ML and dynamics courses.


## Recap

- AI assistants are great for boilerplate, unfamiliar libraries, and
  explaining errors. Always verify.
- Tracebacks read bottom-up. The exception class + message tell you
  what went wrong; the stack tells you where.
- Catch specific exceptions. Don't use `try`/`except` for normal flow.
- `breakpoint()` pauses execution so you can poke.
- `@dataclass` is enough OOP to read library code; deep classes come
  later.
